In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import gaussian_kde

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]
sim_dir = "/Users/z5114326/Documents/simulations/"

mw_gcs_fil = "../data/mw_gcs/mw_gcs.csv"
mw_gcs = pd.read_csv(mw_gcs_fil)

mw_gcs_cols_fil = "../data/mw_gcs/mw_gcs_colors_ref.csv"
mw_gcs_cols = pd.read_csv(mw_gcs_cols_fil)

mw_gc_class = "../data/mw_gcs/class_chen.txt"
mw_class = pd.read_csv(mw_gc_class, sep=r"\s+")

mw_class.loc[mw_class["Cluster"] == "ESO-SC06", "Cluster"] = "ESO_280-SC06"
mw_class.loc[mw_class["Cluster"] == "1636-283", "Cluster"] = "ESO_452-SC11"

mw_class = mw_class.rename(columns={"Cluster": "cluster_name"})

sim_fit_path = "../data/sim_data/boundary_fits.json"
with open(sim_fit_path, "r", encoding="utf-8") as f:
    sim_fit_dict = json.load(f)

sim_fit_upd_path = "../data/sim_data/boundary_fits_updated.json"
with open(sim_fit_upd_path, "r", encoding="utf-8") as f:
    sim_fit_upd_dict = json.load(f)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

In [ ]:
def lower_bound(x, m, b):
    return m * x + b


def kde_threshold(Z, fraction):
    """Return KDE level that encloses fraction of total probability."""
    sorted_Z = np.sort(Z.ravel())[::-1]  # descending
    cumsum = np.cumsum(sorted_Z)
    cumsum /= cumsum[-1]  # normalize
    # Find the smallest Z value where cumulative sum >= fraction
    level = sorted_Z[np.searchsorted(cumsum, fraction)]
    return level


def get_lower_bound(
    x,
    y,
    xlims=(3.0, 8.0),
    ylims=(-1.0, 3.0),
    ngrid=500,
    cont_lvl=0.98,
):

    xlog = np.log10(x)
    ylog = np.log10(y)
    xylog = np.vstack([xlog, ylog])

    xmin, xmax = xlims
    ymin, ymax = ylims

    X, Y = np.meshgrid(
        np.linspace(xmin, xmax, ngrid),
        np.linspace(ymin, ymax, ngrid),
    )

    kde = gaussian_kde(xylog)
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

    lvl = kde_threshold(Z, cont_lvl)

    cont = plt.contour(X, Y, Z, levels=[lvl])
    plt.close()

    contp = cont.get_paths()[0]
    vert = contp.vertices

    xcon = vert[:, 0]
    ycon = vert[:, 1]

    xbins = np.linspace(xcon.min(), xcon.max(), 100)
    xbins_cen = 0.5 * (xbins[:-1] + xbins[1:])

    x_lower = []
    y_lower = []

    for i in range(len(xbins) - 1):
        mask = (xcon >= xbins[i]) & (xcon < xbins[i + 1])

        if np.any(mask):
            x_lower.append(xbins_cen[i])
            y_lower.append(np.min(ycon[mask]))

    x_lower = np.asarray(x_lower)
    y_lower = np.asarray(y_lower)

    p0 = [-1.0, 1.0]
    params, cov = curve_fit(lower_bound, x_lower, y_lower, p0=p0)

    best_fit_dict = {
        "m": params[0],
        "b": params[1],
        "cov": cov,
    }

    xfit = np.logspace(xmin, xmax)
    xfitlog = np.log10(xfit)

    yfitlog = lower_bound(xfitlog, *params)
    yfit = 10**yfitlog

    return xfit, yfit, best_fit_dict


def to_json_serializable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_serializable(v) for v in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    else:
        return obj

In [ ]:
cont_lvl = 0.95
ngrid = 500

xmin, xmax = 3.0, 8.0
ymin, ymax = -1.0, 3.0
X, Y = np.meshgrid(np.linspace(xmin, xmax, ngrid), np.linspace(ymin, ymax, ngrid))

df_orb = mw_gcs.merge(
    mw_class[["cluster_name", "Origin"]],
    on="cluster_name",
    how="inner",
)

rsem = (df_orb["r_apo"] + df_orb["r_per"]) / 2
mini = 10 ** df_orb["log_mini"]
insitu = df_orb["Origin"] == "in-situ"

xi = mini[insitu]
yi = rsem[insitu]
xifit, yifit, fiti_dict = get_lower_bound(xi, yi)

xe = mini[~insitu]
ye = rsem[~insitu]
xefit, yefit, fite_dict = get_lower_bound(xe, ye)

bound_mw = {"in": fiti_dict, "ex": fite_dict}
bound_mw = to_json_serializable(bound_mw)

bound_mw_path = "../data/sim_data/boundary_mw_fits.json"
with open(bound_mw_path, "w", encoding="utf-8") as f:
    json.dump(bound_mw, f, indent=4)

In [ ]:
mi_avg = np.round(fiti_dict["m"], 2)
mi_std = np.round(np.sqrt(fiti_dict["cov"][0, 0]), 2)
bi_avg = np.round(fiti_dict["b"], 2)
bi_std = np.round(np.sqrt(fiti_dict["cov"][1, 1]), 2)

print("MW in situ", "m:", mi_avg, "+/-", mi_std, "b:", bi_avg, "+/-", bi_std)

me_avg = np.round(fite_dict["m"], 2)
me_std = np.round(np.sqrt(fite_dict["cov"][0, 0]), 2)
be_avg = np.round(fite_dict["b"], 2)
be_std = np.round(np.sqrt(fite_dict["cov"][1, 1]), 2)

print("MW ex situ", "m:", me_avg, "+/-", me_std, "b:", be_avg, "+/-", be_std)

In [ ]:
xmin, xmax = (3, 8)
xfitlog = np.linspace(xmin, xmax, 100)
xfit = 10**xfitlog

fig, axs = plt.subplots(2, 1, figsize=(3.5, 4.0))
plt.subplots_adjust(hspace=0.075)

# In-situ population ###########################################

for sim in sim_lst:
    if sim != "m12c":
        m = sim_fit_dict[sim]["in"]["m"]["50"]
        c = sim_fit_dict[sim]["in"]["c"]["50"]

        yfitlog = m * xfitlog + c
        yfit = 10**yfitlog

        axs[0].plot(xfit, yfit, c=sim_colors[sim], lw=0.5, label=sim)
    else:
        m = sim_fit_dict["m12c_5gyr_lim"]["in"]["m"]["50"]
        c = sim_fit_dict["m12c_5gyr_lim"]["in"]["c"]["50"]

        yfitlog = m * xfitlog + c
        yfit = 10**yfitlog

        axs[0].plot(xfit, yfit, c=sim_colors[sim], lw=0.5, ls="--", label="m12c")

axs[0].scatter(xi, yi, c="r", s=10)
axs[0].plot(xifit, yifit, c="r", label="MW In-situ")

axs[0].set_xscale("log")
axs[0].set_yscale("log")

axs[0].set_xlim(7 * 10**3, 7 * 10**6)
axs[0].set_ylim(10**-1, 8 * 10**2)

axs[0].set_xlabel("")
axs[0].set_xticklabels([])

axs[0].set_ylabel(r"a (kpc)")

axs[0].legend(loc="lower left")

# Ex-situ population ###########################################

for sim in sim_lst:
    m = sim_fit_dict[sim]["ex"]["m"]["50"]
    c = sim_fit_dict[sim]["ex"]["c"]["50"]

    yfitlog = m * xfitlog + c
    yfit = 10**yfitlog

    axs[1].plot(xfit, yfit, c=sim_colors[sim], lw=0.5)

axs[1].scatter(xe, ye, c="b", s=10)
axs[1].plot(xefit, yefit, c="b", label="MW Ex-situ")

axs[1].set_xscale("log")
axs[1].set_yscale("log")

axs[1].set_xlim(7 * 10**3, 7 * 10**6)
axs[1].set_ylim(10**-1, 8 * 10**2)

axs[1].set_xlabel(r"$\mathrm{M}_\mathrm{form}\ (\mathrm{M}_\odot)$")
axs[1].set_ylabel(r"a (kpc)")

axs[1].legend(loc="lower left")

mw_conts = "../data/figures/boundaries/mw_kde_boundaries.pdf"
fig.savefig(mw_conts)

# New Boundary Definition

In [ ]:
# xmin, xmax = (3, 8)
# xfitlog = np.linspace(xmin, xmax, 100)
# xfit = 10**xfitlog

# fig, axs = plt.subplots(2, 1, figsize=(3.5, 4.0))
# plt.subplots_adjust(hspace=0.075)

# # In-situ population ###########################################

# for sim in sim_lst:
#     if sim != "m12c":
#         params = (sim_fit_upd_dict[sim]["in"]["avg"]["m"], sim_fit_upd_dict[sim]["in"]["avg"]["b"])

#         yfitlog = lower_bound(xfitlog, *params)
#         yfit = 10**yfitlog

#         axs[0].plot(xfit, yfit, c=sim_colors[sim], lw=1, label=sim)

#     else:
#         params = (
#             sim_fit_upd_dict["m12c_5gyr_lim"]["in"]["avg"]["m"],
#             sim_fit_upd_dict["m12c_5gyr_lim"]["in"]["avg"]["b"],
#         )

#         yfitlog = lower_bound(xfitlog, *params)
#         yfit = 10**yfitlog
#         axs[0].plot(xfit, yfit, c=sim_colors[sim], ls="--", lw=1, label="m12c")


# axs[0].scatter(xi, yi, c="r", s=10)
# axs[0].plot(xifit, yifit, c="r", label="Milky Way")

# axs[0].set_xscale("log")
# axs[0].set_yscale("log")

# axs[0].set_xlim(7 * 10**3, 7 * 10**6)
# axs[0].set_ylim(10**-1, 8 * 10**2)

# axs[0].set_xlabel("")
# axs[0].set_xticklabels([])

# axs[0].set_ylabel(r"a (kpc)")

# axs[0].legend(loc="lower left")

# # Ex-situ population ###########################################

# for sim in sim_lst:
#     params = (sim_fit_upd_dict[sim]["ex"]["avg"]["m"], sim_fit_upd_dict[sim]["ex"]["avg"]["b"])

#     yfitlog = lower_bound(xfitlog, *params)
#     yfit = 10**yfitlog

#     axs[1].plot(xfit, yfit, c=sim_colors[sim], lw=1)

# axs[1].scatter(xe, ye, c="b", s=10)
# axs[1].plot(xefit, yefit, c="b", label="Milky Way")

# axs[1].set_xscale("log")
# axs[1].set_yscale("log")

# axs[1].set_xlim(7 * 10**3, 7 * 10**6)
# axs[1].set_ylim(10**-1, 8 * 10**2)

# axs[1].set_xlabel(r"$\mathrm{M}_\mathrm{form}\ (\mathrm{M}_\odot)$")
# axs[1].set_ylabel(r"a (kpc)")

# axs[1].legend(loc="lower left")

# mw_conts = "../data/figures/boundaries/mw_kde_boundaries_updated.pdf"
# fig.savefig(mw_conts)

In [ ]:
from matplotlib.lines import Line2D

xmin, xmax = (3, 8)
xfitlog = np.linspace(xmin, xmax, 100)
xfit = 10**xfitlog

fig, axs = plt.subplots(2, 1, figsize=(3.5, 4.0))
plt.subplots_adjust(hspace=0.075)

# In-situ population ###########################################

for sim in sim_lst:
    if sim != "m12c":
        params = (
            sim_fit_upd_dict[sim]["in"]["avg"]["m"],
            sim_fit_upd_dict[sim]["in"]["avg"]["b"],
        )

        yfitlog = lower_bound(xfitlog, *params)
        yfit = 10**yfitlog

        axs[0].plot(xfit, yfit, c=sim_colors[sim], lw=0.75, zorder=0)

    else:
        params = (
            sim_fit_upd_dict["m12c_5gyr_lim"]["in"]["avg"]["m"],
            sim_fit_upd_dict["m12c_5gyr_lim"]["in"]["avg"]["b"],
        )

        yfitlog = lower_bound(xfitlog, *params)
        yfit = 10**yfitlog

        axs[0].plot(xfit, yfit, c=sim_colors[sim], ls="--", lw=0.75, zorder=0)

axs[0].scatter(xi, yi, c="r", s=10)
axs[0].plot(xifit, yifit, c="r", lw=2)

axs[0].set_xscale("log")
axs[0].set_yscale("log")

axs[0].set_xlim(7 * 10**3, 7 * 10**6)
axs[0].set_ylim(10**-1, 8 * 10**2)

axs[0].set_xlabel("")
axs[0].set_xticklabels([])

axs[0].set_ylabel(r"a (kpc)")

# MW legend only (upper right)
axs[0].legend(
    handles=[
        Line2D([], [], color="r", lw=2, label="Milky Way"),
    ],
    loc="upper right",
)

# Ex-situ population ###########################################

for sim in sim_lst:
    params = (
        sim_fit_upd_dict[sim]["ex"]["avg"]["m"],
        sim_fit_upd_dict[sim]["ex"]["avg"]["b"],
    )

    yfitlog = lower_bound(xfitlog, *params)
    yfit = 10**yfitlog

    axs[1].plot(xfit, yfit, c=sim_colors[sim], lw=0.75, label=sim, zorder=0)

axs[1].scatter(xe, ye, c="b", s=10)
axs[1].plot(xefit, yefit, c="b", lw=2)

axs[1].set_xscale("log")
axs[1].set_yscale("log")

axs[1].set_xlim(7 * 10**3, 7 * 10**6)
axs[1].set_ylim(10**-1, 8 * 10**2)

axs[1].set_xlabel(r"$\mathrm{M}_\mathrm{form}\ (\mathrm{M}_\odot)$")
axs[1].set_ylabel(r"a (kpc)")

# Simulation legend (bottom left)
sim_legend = axs[1].legend(loc="lower left")

# MW legend (upper right)
mw_legend = axs[1].legend(
    handles=[
        Line2D([], [], color="b", lw=2, label="Milky Way"),
    ],
    loc="upper right",
)

axs[1].add_artist(sim_legend)

mw_conts = "../data/figures/boundaries/mw_kde_boundaries_updated.pdf"
fig.savefig(mw_conts)